# Fine-tuning OWSM with ESPnet3, and decoding with CTC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Courses/CMUSpeechTechnology26S/owsm_finetuning_ctc.ipynb) [![owsm_finetuning_ctc](https://github.com/espnet/notebook/actions/workflows/owsm_finetuning_ctc.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/owsm_finetuning_ctc.yml)

**The badge on this notebook is a short run.** The weekly job fine-tunes for
two steps on a handful of utterances, because a free runner cannot do more.
Green means this notebook still installs, still finds its data and still
agrees with ESPnet3's API — not that the fine-tuning below was reproduced.
Open it and you get the full run.

Fine-tune a pretrained OWSM model on a small spoken-digit corpus with the
ESPnet3 trainer, then compare autoregressive decoding against CTC greedy
decoding, in word error rate and in time.

This is the demonstration part of an assignment from CMU 11492/11692/18495,
*Speech Technology for Conversational AI*, kept here without the graded
exercises so that it runs end to end.

Author: Siqi Ouyang (siqiouya@andrew.cmu.edu)

Main references:
- [ESPnet repository](https://github.com/espnet/espnet)
- [ESPnet3 recipes](https://github.com/espnet/espnet/tree/master/egs3)
- [OWSM](https://www.wavlab.org/activities/2024/owsm/)

## 1) Prerequisites (3 minutes)

### Environment setup

- This is a full installation method to perform data preprocessing, training and inference.

- We prepare various ways of installation. Please read https://espnet.github.io/espnet/installation.html#step-2-installation-espnet for more details.

- We also have some other toolkits/packages needed for this assignment.

The training stack comes with the `[train]` extra: Lightning, Hydra and
`datasets`, which is what ESPnet3's trainer is built on. `jiwer` is for the
word error rates further down.

The pin matters more here than in an inference notebook. Fine-tuning reads a
config that the trainer's API has to agree with, and a release is a fixed
agreement; `master` is not.

In [ ]:
%pip install -q "espnet[train]==202610.post1" jiwer

## 2) Imports

### Runtime imports


In [ ]:
import os
import time
from pathlib import Path

import jiwer
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from omegaconf import OmegaConf
from espnet2.bin.s2t_inference import Speech2Text

## 3) Data

The course used TIDIGITS, which is licensed by the LDC and cannot be
redistributed, so this version uses [Google Speech
Commands](https://arxiv.org/abs/1804.03209) v0.02 instead (CC-BY 4.0). We keep
twelve words - *yes*, *no* and the digits *zero* to *nine* - which is the same
kind of task: a tiny vocabulary where fine-tuning shows up clearly in the word
error rate.

The archive holds 35 words and is 2.3 GB, so rather than downloading it and
throwing most of it away, the cell below streams it and writes only the clips
it wants. That takes about a minute. The recordings are 16 kHz mono, which is
what OWSM expects.


### Download and split

The split is by speaker, not by clip: the same person saying *nine* twice must
not land in both training and test, or the word error rate afterwards means
nothing.


In [ ]:
import collections
import pathlib
import tarfile
import urllib.request

SPEECH_COMMANDS = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
WORDS = ["yes", "no"] + "zero one two three four five six seven eight nine".split()
# Fine-tuning is the one thing in these notebooks that a free CI runner
# cannot do at full size, so the weekly run shrinks it through the
# environment. Unset - which is what you have when you open this - each of
# these is the real thing.
PER_WORD = int(os.environ.get("PER_WORD", 60))  # 60 is enough to see it work
LIMIT_TRAIN_BATCHES = int(os.environ.get("LIMIT_TRAIN_BATCHES", 300))
MAX_TEST_CLIPS = int(os.environ.get("MAX_TEST_CLIPS", 0))  # 0 is all of them

DATA_ROOT = pathlib.Path("speech_commands")
kept = collections.Counter()
if not DATA_ROOT.exists():
    # stream the archive and keep only the words we asked for, instead of
    # downloading 2.3 GB to disk and deleting most of it
    with urllib.request.urlopen(SPEECH_COMMANDS) as response, tarfile.open(
        fileobj=response, mode="r|gz"
    ) as tar:
        for member in tar:
            if not member.isfile() or not member.name.endswith(".wav"):
                continue
            word = member.name.split("/")[-2]
            if word not in WORDS or kept[word] >= PER_WORD:
                continue
            tar.extract(member, DATA_ROOT, filter="data")
            kept[word] += 1
            if len(kept) == len(WORDS) and all(n >= PER_WORD for n in kept.values()):
                break


### Custom dataset

We need to convert the clips into a format that ESPnet can read: a PyTorch
`Dataset` whose items carry the fields OWSM's preprocessor expects.

**It goes in a file, not in this notebook.** ESPnet3 does not take a dataset
object: `DataOrganizer` resolves each entry in the config through
`load_dataset_module()`, which imports `<recipe_dir>/dataset/__init__.py` and
looks for a class called `Dataset`. The config then names that class's
arguments under `data_src_args`. A recipe in `egs3/` is laid out the same way,
so what you write here is what you would write there.

Related docs:
- [**Create dataset stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/create-dataset.html)
- [**Trainer config**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/config/train_config.html)
- [**Data organizer**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/core/components/data-organizer.html)

In [ ]:
from pathlib import Path

Path("dataset").mkdir(exist_ok=True)

In [ ]:
%%writefile dataset/__init__.py
"""The clips, in the fields ESPnet's S2T preprocessor expects.

ESPnet3 imports this file: the class has to be called `Dataset`, and its
arguments are what the training config passes as `data_src_args`. The split
lives here too, so that the notebook and the trainer cannot disagree about
which speaker is in which split.
"""

import pathlib

import numpy as np
import soundfile
from torch.utils.data import Dataset as TorchDataset


def splits(data_root):
    """The clips under `data_root`, divided by speaker.

    By speaker and not by clip: the same person saying "nine" twice must not
    land in both training and test, or the word error rate afterwards means
    nothing. The speaker is the filename prefix.
    """
    clips = sorted(pathlib.Path(data_root).glob("*/*.wav"))
    speaker_of = lambda path: path.name.split("_")[0]  # noqa: E731
    speakers = sorted({speaker_of(p) for p in clips})
    held_out = set(speakers[: len(speakers) // 5])            # a fifth for test
    validation = set(speakers[len(speakers) // 5 : len(speakers) // 4])
    divided = {"train": [], "validation": [], "test": []}
    for clip in clips:
        who = speaker_of(clip)
        name = (
            "test" if who in held_out
            else "validation" if who in validation
            else "train"
        )
        divided[name].append(clip)
    return divided


class Dataset(TorchDataset):
    """The transcript of a clip is the word it belongs to, which is the name
    of the directory holding it."""

    def __init__(self, *, data_root: str, split: str, lang_sym: str,
                 task_sym: str) -> None:
        self.lang_sym = str(lang_sym)
        self.task_sym = str(task_sym)
        self._clips = splits(data_root)[str(split)]

    def __len__(self) -> int:
        return len(self._clips)

    def __getitem__(self, idx: int):
        clip = self._clips[int(idx)]
        speech, _ = soundfile.read(clip, dtype="float32")
        transcription = clip.parent.name
        text = f"{self.lang_sym}{self.task_sym}<notimestamps> {transcription}"
        return {
            "speech": speech.astype(np.float32),
            "text": text,
            "text_prev": "<na>",
            "text_ctc": transcription,
        }

In [ ]:
from dataset import splits

SPLITS = splits(DATA_ROOT)
print({name: len(paths) for name, paths in SPLITS.items()})

### Inspect the Data


In [ ]:
clip = SPLITS["train"][0]
print("word:", clip.parent.name)
print("speaker:", clip.name.split("_")[0])
print("file:", clip)


In [ ]:
import soundfile
from IPython.display import Audio, display

clip = SPLITS["train"][0]
wav, fs = soundfile.read(clip, dtype="float32")
print(clip.parent.name, wav.shape, fs)
display(Audio(wav, rate=fs))


## 4) Pre-Trained Model

In low-resource settings, training a model from scratch is unlikely to lead to good results. So instead, we will fine-tune a pre-trained foundation model.
We will use the base version of [OWSM 3.1](https://arxiv.org/pdf/2401.16658), an open-source speech foundation model trained on 180K hours of multilingual ASR and ST.

Here we also set the path `WORK_DIR`, in which dataset, checkpoints and logs will be saved.


### Downloading

Since it needs to support many language varieties, OWSM uses ISO3 for the language IDs. The ISO3 code for your language of choice can also be found in Table 9 in the FLEURS paper: https://arxiv.org/pdf/2205.12446


In [ ]:
MODEL_TAG = "espnet/owsm_v3.1_ebf_base"     # You can also change to OWSM v4: "espnet/owsm_v4_base_102M"
OWSM_LANG = "eng"  # ISO3 (e.g., eng, jpn)

In [ ]:
WORK_DIR = Path(os.environ.get("WORK_DIR", "./work/owsm_v31_tidigits")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

s2t = Speech2Text.from_pretrained(
    model_tag=MODEL_TAG,
    lang_sym=f"<{OWSM_LANG}>",
    device=DEVICE,
    beam_size=1,  # to align with CTC greedy decoding
)
torch.save(s2t.s2t_model.state_dict(), 'original.pth')

BPEMODEL = s2t.tokenizer.model
TOKEN_LIST = WORK_DIR / "token_list.txt"
TOKEN_LIST.write_text("\n".join(s2t.converter.token_list))

tokenizer = s2t.tokenizer
converter = s2t.converter

def tokenize(text):
    return np.array(converter.tokens2ids(tokenizer.text2tokens(text)))

def detokenize(ids):
    return tokenizer.tokens2text(converter.ids2tokens(ids))


### Custom model wrapper

We need to define a class that will pass our pre-trained model to ESPnet

Related docs:
- [**Model component**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/core/components/model.html)


In [ ]:
class OWSMBaseFinetuneModel(nn.Module):
    def __init__(
        self,
        *,
        model_tag: str,
        lang_sym: str,
        device: str = "cpu",
        ctc_weight: float,
        specaug: nn.Module = None,
    ) -> None:
        super().__init__()
        s2t = Speech2Text.from_pretrained(
            model_tag=model_tag,
            lang_sym=lang_sym,
            device=str(device),
        )
        self.s2t_model = s2t.s2t_model
        self.s2t_model.ctc_weight = ctc_weight
        self.s2t_model.specaug = specaug

    def forward(self, **batch):
        return self.s2t_model(**batch)

    def collect_feats(self, **batch):
        return self.s2t_model.collect_feats(**batch)


## 5) Training config in YAML

### Config

Training requires tuning many hyper-parameters. We have provided an initial config here to start you off.

We also need to pass our custom model class `__main__.OWSMBaseFinetuneModel` and dataset class `__main__.SpeechCommandsDataset` to the config.

<!-- Edit `max_epochs`, `batch_bins`, or `limit_train_batches` for quick tests.-->

Related docs:
- [**Systems overview**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/core/systems.html)
- [**Train stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/train.html)
- [**Training config**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/config/train_config.html)


In [ ]:
EXP_TAG = f"owsm_v31_base_tidigits"
EXP_DIR = (WORK_DIR / "exp" / EXP_TAG).as_posix()
STATS_DIR = (WORK_DIR / "exp" / "stats").as_posix()

LANG_SYM = f"<{OWSM_LANG}>"
TASK_SYM = "<asr>"

yaml_cfg = f"""
num_device: 1
num_nodes: 1
exp_tag: {EXP_TAG}
recipe_dir: .
data_dir: {WORK_DIR / 'data'}
exp_dir: {EXP_DIR}
stats_dir: {STATS_DIR}
dataset_dir: {WORK_DIR / 'hf_cache'}

dataset:
  _target_: espnet3.components.data.data_organizer.DataOrganizer
  # where the organizer looks for dataset/__init__.py, whose Dataset class it
  # builds with each entry's data_src_args
  recipe_dir: .
  train:
    - name: train
      data_src_args:
        data_root: {DATA_ROOT}
        split: train
        lang_sym: {LANG_SYM}
        task_sym: {TASK_SYM}
  valid:
    - name: validation
      data_src_args:
        data_root: {DATA_ROOT}
        split: validation
        lang_sym: {LANG_SYM}
        task_sym: {TASK_SYM}
  preprocessor:
    _target_: espnet2.train.preprocessor.S2TPreprocessor
    train: true
    token_type: bpe
    token_list: {TOKEN_LIST}
    bpemodel: {BPEMODEL}
    text_prev_name: text_prev
    text_ctc_name: text_ctc
    fs: 16000
    data_aug_prob: 0.0 # the probability to apply data augmentation
    data_aug_effects: # data augmentation effects (speed perturbation)
      - - 1.0
        - - [0.5, "speed_perturb", {{"factor": 0.9}}]
          - [0.5, "speed_perturb", {{"factor": 1.1}}]

dataloader:
  collate_fn:
    _target_: espnet2.train.collate_fn.CommonCollateFn
    int_pad_value: -1
  train:
    iter_factory:
      _target_: espnet2.iterators.sequence_iter_factory.SequenceIterFactory
      shuffle: true
      collate_fn: ${{dataloader.collate_fn}}
      num_workers: 0
      batches:
        type: numel
        shape_files:
          - ${{stats_dir}}/train/feats_shape # the length of each data sample
        batch_size: 2
        batch_bins: 1000000
  valid:
    iter_factory:
      _target_: espnet2.iterators.sequence_iter_factory.SequenceIterFactory
      shuffle: false
      collate_fn: ${{dataloader.collate_fn}}
      num_workers: 0
      batches:
        type: numel
        shape_files:
          - ${{stats_dir}}/valid/feats_shape
        batch_size: 2
        batch_bins: 2000000

model:
  _target_: __main__.OWSMBaseFinetuneModel
  model_tag: {MODEL_TAG}
  lang_sym: {LANG_SYM}
  device: {DEVICE}
  ctc_weight: 0.3
  specaug: # SpecAugment configuration
    _target_: espnet2.asr.specaug.specaug.SpecAug
    apply_time_warp: false
    time_warp_window: 5
    time_warp_mode: bicubic
    apply_freq_mask: true
    freq_mask_width_range:
      - 0
      - 27
    num_freq_mask: 1
    apply_time_mask: true
    time_mask_width_ratio_range:
      - 0.0
      - 0.05
    num_time_mask: 1

optimizer:
  _target_: torch.optim.Adam
  lr: 3.0e-5

scheduler:
  _target_: torch.optim.lr_scheduler.StepLR
  step_size: 1000

best_model_criterion:
  - - valid/acc
    - 3
    - max

trainer:
  accelerator: auto
  devices: 1
  max_epochs: 1
  log_every_n_steps: 1
  limit_train_batches: {LIMIT_TRAIN_BATCHES}

fit: {{}}
"""

cfg = OmegaConf.create(yaml_cfg)
print("Config ready. exp_dir:", cfg.exp_dir)

🔍 **Possible Exploration:** Adjust following hyperparameters to find out how it affects model training and inference performance.
- `dataset.preprocessor.data_aug_prob`: increase the probability of applying speed perturbation (by default, it is 0.0, meaning it's turned off)
- `model.ctc_weight`: the weight of CTC loss in CTC/attention hybrid training
- `model.specaug`: adjust the SpecAugment configuration

Don't forget to modify `EXP_TAG` and `EXP_DIR` to save results in different directories.


## 6) Train (5-6 minutes)

### collect_stats + train

Finally, we pass the config to the trainer and start training.
ESPnet3’s training framework is built on top of the PyTorch Lightning Trainer.

Related docs:
- [**Collect Stats stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/collect-stats.html)
- [**Train stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/train.html)
- [**Training config**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/config/train_config.html)


In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

# Launch tensorboard before training
%tensorboard --logdir lightning_logs/

In [ ]:
from espnet3.systems.base.system import BaseSystem
from espnet3.utils.logging_utils import configure_logging
from espnet3.utils.stages_utils import run_stages

log = configure_logging()
system = BaseSystem(training_config=cfg)

# collect_stats for efficient batching
run_stages(system, ["collect_stats", "train"], log=log)

## 7) Inference

Here is a demo of how to perform inference, and how to load checkpoints.


### Autoregressive decoding with original pre-trained model


In [ ]:
# the same class the trainer used, imported from the module it lives in
from dataset import Dataset as SpeechCommandsDataset
from torch.utils.data import Subset

test_dataset = SpeechCommandsDataset(
    data_root=DATA_ROOT, split="test", lang_sym=LANG_SYM, task_sym=TASK_SYM
)
if MAX_TEST_CLIPS:
    test_dataset = Subset(test_dataset, range(min(MAX_TEST_CLIPS, len(test_dataset))))
print(len(test_dataset), "test clips")
sample_test_utterance = test_dataset[0]

In [ ]:
import torch
_device="cuda" if torch.cuda.is_available() else "cpu" if torch.cuda.is_available() else "cpu"
s2t.s2t_model.to(_device)
s2t.device = _device

d = torch.load("original.pth", map_location='cpu')
s2t.s2t_model.load_state_dict(d)
pred = s2t(sample_test_utterance['speech'])
print('PREDICTED: ' + pred[0][0])
print('REFERENCE: ' + sample_test_utterance['text_ctc'])

### Autoregressive decoding with fine-tuned model


In [ ]:
# the last checkpoint the trainer wrote, rather than a step number spelled
# out here: the number follows limit_train_batches and max_epochs
checkpoint = max(
    Path(EXP_DIR).glob("step*.ckpt"), key=lambda p: int(p.stem.removeprefix("step"))
)
d = torch.load(checkpoint, map_location="cpu")
s2t.s2t_model.load_state_dict({
    k.replace("s2t_model.", "", 1): v
    for k, v in d["state_dict"].items()
})
pred = s2t(sample_test_utterance['speech'])
print('PREDICTED: ' + pred[0][0])
print('REFERENCE: ' + sample_test_utterance['text_ctc'])

### CTC decoding with fine-tuned model


In [ ]:
# best_path is the CTC head of the model already loaded above: greedy, no
# search, no second copy of the weights in memory. Calling s2t instead runs
# the beam search over the decoder, which is the other half of this comparison.
pred = s2t.best_path(sample_test_utterance['speech'])
print('PREDICTED: ' + pred[0][0])
print('REFERENCE: ' + sample_test_utterance['text_ctc'])

## 8) WER Calculation


### Autoregressive decoding with original pre-trained model


In [ ]:
hyps = []
refs = []
d = torch.load("original.pth", map_location='cpu') # Default to ckpt before fine-tuning.
s2t.s2t_model.load_state_dict(d)

for sample in test_dataset:
  hyp = s2t(sample['speech'])[0][3] # decoded transcript without special tokens
  refs.append(sample['text_ctc'])
  hyps.append(hyp)

In [ ]:
# Compute WER and CER for multiple examples
ar_original_wer_scores = [jiwer.wer(ref, hyp) for ref, hyp in zip(refs, hyps)]
ar_original_cer_scores = [jiwer.cer(ref, hyp) for ref, hyp in zip(refs, hyps)]

# Compute Average WER and CER
average_wer = sum(ar_original_wer_scores) / len(ar_original_wer_scores)
average_cer = sum(ar_original_cer_scores) / len(ar_original_cer_scores)

print(f"Average WER: {average_wer:.2%}")
print(f"Average CER: {average_cer:.2%}")


### Autoregressive decoding with fine-tuned model


In [ ]:
hyps = []
refs = []
# the last checkpoint the trainer wrote, rather than a step number spelled
# out here: the number follows limit_train_batches and max_epochs
checkpoint = max(
    Path(EXP_DIR).glob("step*.ckpt"), key=lambda p: int(p.stem.removeprefix("step"))
)
d = torch.load(checkpoint, map_location="cpu")
s2t.s2t_model.load_state_dict({
    k.replace("s2t_model.", "", 1): v
    for k, v in d["state_dict"].items()
})

for sample in test_dataset:
  hyp = s2t(sample['speech'])[0][3] # decoded transcript without special tokens
  refs.append(sample['text_ctc'])
  hyps.append(hyp)

In [ ]:
# Compute WER and CER for multiple examples
ar_ft_wer_scores = [jiwer.wer(ref, hyp) for ref, hyp in zip(refs, hyps)]
ar_ft_cer_scores = [jiwer.cer(ref, hyp) for ref, hyp in zip(refs, hyps)]

# Compute Average WER and CER
average_wer = sum(ar_ft_wer_scores) / len(ar_ft_wer_scores)
average_cer = sum(ar_ft_cer_scores) / len(ar_ft_cer_scores)

print(f"Average WER: {average_wer:.2%}")
print(f"Average CER: {average_cer:.2%}")


### CTC decoding with fine-tuned model


In [ ]:
hyps = []
refs = []
for sample in test_dataset:
  hyp = s2t.best_path(sample['speech'])[0][3] # decoded transcript without special tokens
  refs.append(sample['text_ctc'])
  hyps.append(hyp)

In [ ]:
# Compute WER and CER for multiple examples
ctc_ft_wer_scores = [jiwer.wer(ref, hyp) for ref, hyp in zip(refs, hyps)]
ctc_ft_cer_scores = [jiwer.cer(ref, hyp) for ref, hyp in zip(refs, hyps)]

# Compute Average WER and CER
average_wer = sum(ctc_ft_wer_scores) / len(ctc_ft_wer_scores)
average_cer = sum(ctc_ft_cer_scores) / len(ctc_ft_cer_scores)

print(f"Average WER: {average_wer:.2%}")
print(f"Average CER: {average_cer:.2%}")

In [ ]:
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.perf_counter()

for sample in test_dataset:
  s2t(sample['speech'])

torch.cuda.synchronize() if torch.cuda.is_available() else None
print(f"Autoregressive decoding time: {time.perf_counter() - t0:.2f}s")

In [ ]:
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.perf_counter()

for sample in test_dataset:
  s2t.best_path(sample['speech'])

torch.cuda.synchronize() if torch.cuda.is_available() else None
print(f"CTC decoding time: {time.perf_counter() - t0:.2f}s")

In [ ]:
d = torch.load("original.pth", map_location='cpu')
s2t.s2t_model.load_state_dict(d)

cfg.optimizer.lr = 1e-3

# save the experiment result to a new directory
EXP_TAG_lr = EXP_TAG + "_lr1e-3"
EXP_DIR_lr = EXP_DIR + "_lr1e-3"
cfg.exp_tag = EXP_TAG_lr
cfg.exp_dir = EXP_DIR_lr

system = BaseSystem(training_config=cfg)
run_stages(system, ["collect_stats", "train"], log=log)

In [ ]:
hyps = []
refs = []
# the last checkpoint the trainer wrote, rather than a step number spelled
# out here: the number follows limit_train_batches and max_epochs
checkpoint = max(
    Path(EXP_DIR_lr).glob("step*.ckpt"), key=lambda p: int(p.stem.removeprefix("step"))
)
d = torch.load(checkpoint, map_location="cpu")
s2t.s2t_model.load_state_dict({
    k.replace("s2t_model.", "", 1): v
    for k, v in d["state_dict"].items()
})

for sample in test_dataset:
  hyp = s2t(sample['speech'])[0][3] # decoded transcript without special tokens
  refs.append(sample['text_ctc'])
  hyps.append(hyp)

In [ ]:
# Compute WER and CER for multiple examples
wer_scores = [jiwer.wer(ref, hyp) for ref, hyp in zip(refs, hyps)]
cer_scores = [jiwer.cer(ref, hyp) for ref, hyp in zip(refs, hyps)]

# Compute Average WER and CER
average_wer = sum(wer_scores) / len(wer_scores)
average_cer = sum(cer_scores) / len(cer_scores)

print(f"Average WER: {average_wer:.2%}")
print(f"Average CER: {average_cer:.2%}")